# Gene_selection_stability_ROUND4

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
# NOTE: deliberately NOT calling warnings.filterwarnings("ignore") anywhere
# in this notebook. Warnings are handled explicitly, per resample, below.
import pandas as pd
import numpy as np
import time
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
import warnings

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
N_BOOTSTRAP = 200

# Exact, unrounded alpha values from the original locked derivation
ALPHA_MIN = 0.039161227341523885
ALPHA_1SE = 0.06532486436580959
print(f"Exact alpha values: alpha_min={ALPHA_MIN}, alpha_1se={ALPHA_1SE}")

In [ ]:
frozen = pd.read_csv("FROZEN_candidate_pool.csv")
full_coverage = pd.read_csv("full_candidate_pool_final_coverage.csv")
frozen_genes = frozen["Gene"].tolist()

sub = full_coverage[full_coverage["Gene"].isin(frozen_genes)]
gene_to_col = {}
for _, r in sub.iterrows():
    gene_to_col[r["Gene"]] = r["Gene"] if r["TCGA_HNSC_Match"] == "Direct" else r["TCGA_HNSC_Alias_Used"]

genes = list(gene_to_col.keys())
tcga_cols = list(gene_to_col.values())
print("Candidate pool genes:", len(gene_to_col))

In [ ]:
usecols = ["Patient ID", "Sample ID", "Overall Survival (Months)", "Overall Survival Status"] + tcga_cols
df = pd.read_csv("clinical_rna_fire_combine.csv", usecols=usecols, engine="python")
df = df[df["Sample ID"].str.endswith("-01")].copy()
for c in tcga_cols:
    df[c] = np.log2(df[c].astype(float) + 1)
df["OS_months"] = df["Overall Survival (Months)"]
df["Event"] = df["Overall Survival Status"].astype(str).str.startswith("1").astype(int)
df = df.dropna(subset=["OS_months", "Event"])
df = df[df["OS_months"] > 0].reset_index(drop=True)
print("Derivation cohort: N =", df.shape[0], "| Events =", int(df["Event"].sum()))

X_raw = df[tcga_cols].copy()
X_raw.columns = genes

locked_min_genes = set(pd.read_csv("HNSC_hypoxia_signature_lambda_min.csv")["Gene"])
locked_1se_genes = set(pd.read_csv("HNSC_hypoxia_signature_lambda_1se.csv")["Gene"])

In [ ]:
def run_bootstrap_stability(alpha, label, n_boot=N_BOOTSTRAP):
    print(f"Running {n_boot} bootstrap resamples for {label} (exact alpha={alpha})...")
    gene_selection_counts = {g: 0 for g in genes}
    n_successful = 0
    n_failed = 0
    failure_log = []
    all_captured_warnings = []
    start = time.time()

    rng = np.random.RandomState(RANDOM_STATE)
    for b in range(n_boot):
        boot_idx = rng.choice(len(X_raw), size=len(X_raw), replace=True)
        X_boot_raw = X_raw.iloc[boot_idx].reset_index(drop=True)
        y_boot = Surv.from_dataframe("Event", "OS_months", df.iloc[boot_idx].reset_index(drop=True))

        X_boot = (X_boot_raw - X_boot_raw.mean()) / X_boot_raw.std()
        X_boot = X_boot.fillna(0)

        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")  # override any outer filter for this block only
            try:
                model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[alpha])
                model.fit(X_boot.values, y_boot)
                coefs = model.coef_.flatten()
                for g, c in zip(genes, coefs):
                    if c != 0:
                        gene_selection_counts[g] += 1
                n_successful += 1
            except Exception as e:
                n_failed += 1
                failure_log.append(f"Resample {b}: {type(e).__name__}: {str(e)[:150]}")

            # Record EVERY warning caught in this block, regardless of category
            for warning_item in w:
                all_captured_warnings.append(
                    f"Resample {b}: {warning_item.category.__name__}: {str(warning_item.message)[:200]}"
                )

    elapsed = time.time() - start
    print(f"Completed: {n_successful}/{n_boot} successful, {n_failed}/{n_boot} failed, {elapsed:.0f}s")
    print(f"Total warnings captured (any category): {len(all_captured_warnings)}")
    if failure_log:
        print("FAILURES:")
        for f in failure_log:
            print(f"  {f}")
    if all_captured_warnings:
        print("WARNINGS CAPTURED (full list):")
        for wmsg in all_captured_warnings:
            print(f"  {wmsg}")
    else:
        print("No warnings of any category were captured across all resamples.")

    stability_df = pd.DataFrame([
        {"Gene": g, "Selection_frequency": gene_selection_counts[g] / n_successful}
        for g in genes
    ]).sort_values("Selection_frequency", ascending=False)

    return stability_df, n_successful, n_failed, failure_log, all_captured_warnings

In [ ]:
stability_min, n_min, nfail_min, flog_min, wlog_min = run_bootstrap_stability(ALPHA_MIN, "lambda.min (32 genes)")
stability_min.to_csv("stability_lambda_min_ROUND4.csv", index=False)

In [ ]:
stability_1se, n_1se, nfail_1se, flog_1se, wlog_1se = run_bootstrap_stability(ALPHA_1SE, "lambda.1se (13 genes)")
stability_1se.to_csv("stability_lambda_1se_ROUND4.csv", index=False)

In [ ]:
locked_min_stability = stability_min[stability_min["Gene"].isin(locked_min_genes)].sort_values("Selection_frequency", ascending=False)
print("=== lambda.min locked genes ===")
print(locked_min_stability.to_string(index=False))
print(f"Mean: {locked_min_stability['Selection_frequency'].mean():.3f}")

In [ ]:
locked_1se_stability = stability_1se[stability_1se["Gene"].isin(locked_1se_genes)].sort_values("Selection_frequency", ascending=False)
print("=== lambda.1se locked genes ===")
print(locked_1se_stability.to_string(index=False))
print(f"Mean: {locked_1se_stability['Selection_frequency'].mean():.3f}")

In [ ]:
print("=== FINAL VERIFIED SUMMARY (no warnings suppressed anywhere in this notebook) ===")
print(f"lambda.min:  {n_min}/{N_BOOTSTRAP} successful, {nfail_min} failed, {len(wlog_min)} warnings captured")
print(f"lambda.1se:  {n_1se}/{N_BOOTSTRAP} successful, {nfail_1se} failed, {len(wlog_1se)} warnings captured")

import json
with open("stability_summary_ROUND4.json", "w") as f:
    json.dump({
        "alpha_min": ALPHA_MIN, "alpha_1se": ALPHA_1SE, "n_bootstrap": N_BOOTSTRAP,
        "n_successful_min": n_min, "n_failed_min": nfail_min, "warnings_min": wlog_min,
        "n_successful_1se": n_1se, "n_failed_1se": nfail_1se, "warnings_1se": wlog_1se,
    }, f, indent=2)
print("\nSaved stability_summary_ROUND4.json")